In [11]:
import logging

import pandas as pd

import data.helpers as dh
import src.models.helpers as mh
import src.cfr.cfr_helpers as cfrh

import data.cfr_data_19_23 as cfrd
import data.breathe_data as bd
import datetime

Exploring the CF Trust registry data from 2019 to 2023 to evaluate if model output (synthesizing FEV1, FEF25-75 on 2 days) can be used to improve ML achieved from each indidivually 

# Load, process, save yearly data

In [2]:
# Load 2019 data
df19 = cfrd.build_cfr_df(2019)
# WARNINGS:
# Surprisingly low pFEV1 are old females (69-80)
# Surprisingly high pFEV1 is a tall man (1.93m)

KeyboardInterrupt: 

In [4]:
df19old = bd.load_meas_from_excel("CF_Registry_19_processed", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [4]:
df19.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_processed.xlsx",
    index=False,
)

In [2]:
df23 = cfrd.build_cfr_df(2023)

INFO:root:Loaded 10344 entries
INFO:root:5065 after removing all NaN
INFO:root:2701 entries after removing <18yr


In [7]:
df23.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_23_processed.xlsx",
    index=False,
)

# Load data

## Link 2019 with 2023 data

In [ ]:
df19 = bd.load_meas_from_excel("CF_Registry_19_processed", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [ ]:
df = pd.concat([df19, df23]).sort_values("ID")

In [ ]:
(df.groupby("ID").apply(lambda df: len(df)) > 1).sum()

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_32318/3045146862.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  (df.groupby('ID').apply(lambda df: len(df)) > 1).sum()


1485

In [ ]:
df = bd.load_meas_from_excel("CF_Registry_19_processed_with_idx", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [ ]:
ids_with_2_entries = (
    df["ID"].value_counts()[df["ID"].value_counts() == 2].index.tolist()
)
df = df[df.ID.isin(ids_with_2_entries)]

df.to_excel(
    dh.get_path_to_main()
    + "ExcelFiles/CFR/CF_Registry_19_23_stricly_2_entries_processed_with_idx.xlsx",
    index=False,
)

## Use 2019 / 2023 data only

In [2]:
cols2read = [
    "s01caseid_original",
    # "s01sex",
    "s01height",
    "s01encounterageyears",
    "s03cliqtrfev1",  # Value at annual review
    "s03clibestfev1",
    "s03clifef2575",  # Value at annual review
]
colnames = ["ID", "Height", "Age", "FEV1", "best FEV1", "FEF2575"]

df = cfrd.build_cfr_df(2019, cols2read, colnames)

KeyboardInterrupt: 

In [9]:
df

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted,best FEV1 old,idx FEV1,idx FEF2575%FEV1,idx best FEV1
0,B155916,32,162,1.50,0.47,1.64,Female,2019-01-01,1.50,0.47,31.333333,3.158943,47.484238,47.484238,1.64,30,15,32
1,B155917,45,175,2.67,1.00,2.67,Male,2019-01-01,2.67,1.00,37.453182,3.937246,67.813909,67.813909,2.67,53,18,53
2,B155918,34,191,4.82,3.48,4.99,Male,2019-01-01,4.82,3.48,72.199168,5.141756,93.742289,93.742289,4.99,96,36,99
3,B155921,34,150,1.44,0.59,1.45,Female,2019-01-01,1.44,0.59,40.972219,2.653810,54.261617,54.261617,1.45,28,20,29
4,B155925,38,167,0.92,0.34,1.33,Female,2019-01-01,0.92,0.34,36.956521,3.244338,28.357098,28.357098,1.33,18,18,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,C222738,59,158,1.66,1.18,1.95,Female,2019-01-01,1.66,1.18,71.084336,2.377677,69.816049,69.816049,1.95,33,35,39
2033,C222739,69,153,1.63,0.66,1.63,Female,2019-01-01,1.63,0.66,40.490799,1.968185,82.817413,82.817413,1.63,32,20,32
2034,C222741,33,162,1.71,0.86,1.77,Female,2019-01-01,1.71,0.86,50.292397,3.142266,54.419331,54.419331,1.77,34,25,35
2035,C222780,27,176,3.54,4.15,3.54,Female,2019-01-01,3.54,4.15,117.231642,3.851176,91.919978,91.919978,3.54,70,58,70


In [ ]:
df["best FEV1 old"] = df["best FEV1"]
df["best FEV1"] = df["best FEV1"].where(df["best FEV1"] >= df["FEV1"], df["FEV1"])

In [ ]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_processed_with_idx.xlsx",
    index=False,
)

# Prep running inference

## Add obs indices

In [10]:
# Add indices for model
# height = df.Height.iloc[0]
# age = df.Age.iloc[0]
# sex = df.Sex.iloc[0]
# ar_prior = "uniform"
# ecfev1_noise_model_cpt_suffix = "_std_add_mult_ecfev1"
# ar_fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"
# (
#     HFEV1,
#     uFEV1,
#     ecFEV1,
#     AR,
#     ecFEF2575prctecFEV1,
# ) = var_builders.fev1_fef2575_point_in_time_model_noise_shared_healthy_vars(
#     height,
#     age,
#     sex,
#     ar_prior,
#     ecfev1_noise_model_cpt_suffix,
#     ar_fef2575_cpt_suffix,
# )

import src.models.helpers as mh

ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
ecFEF2575prctecFEV1 = mh.VariableNode("ecFEF25-75 % ecFEV1 (%)", 0, 200, 2, prior=None)

# df[f"idx {ecFEV1.name}"] = df.apply(
#     lambda row: ecFEV1.get_bin_idx_for_value(row["ecFEV1"]), axis=1
# )
# df[f"idx {ecFEF2575prctecFEV1.name}"] = df.apply(
#     lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
#     axis=1,
# )
df[f"idx FEV1"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["ecFEV1"]), axis=1
)
df[f"idx FEF2575%FEV1"] = df.apply(
    lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
    axis=1,
)
df[f"idx best FEV1"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["best FEV1"]), axis=1
)

## Custom inference (for 2019 or 2023 only data)

In [8]:
df = bd.load_meas_from_excel("infer_all_19_data_with_best_FEV1", study_folder="CFR")

print(f"Initial shape {df.shape}")
df.dropna(subset=["FEV1", "FEF2575", "best FEV1"])
print(f"Final shape {df.shape} - Any rows dropped?")

INFO:root:* Checking for same day measurements *


Initial shape (2037, 22)
Final shape (2037, 22) - Any rows dropped?


In [9]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1', 'idx best FEV1',
       'P(HFEV1|FEF2575, bFEV1)', 'P(FEV1|HFEV1_pers)', 'P(FEV1_obs|FEV1)',
       'Airway resistance (%)'],
      dtype='object')

In [ ]:
dftmp = df
dftmp["P(HFEV1|FEF2575, bFEV1, FEV1)"] = dftmp.apply(cfrh.infer_hfev1_pers, axis=1)
dftmp["P(HFEV1|FEV1)"] = dftmp.apply(cfrh.infer_hfev1_soft_truncation, axis=1)
# dftmp["P(FEV1|HFEV1_pers)"] = dftmp.apply(cfrh.infer_fev1_pred, axis=1)
# dftmp["P(FEV1_obs|FEV1)"] = dftmp.apply(
#     lambda row: row["P(FEV1|HFEV1_pers)"][row["idx FEV1"]], axis=1
# )

In [5]:
AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, prior={"type": "uniform"})
dftmp[AR.name] = dftmp.apply(cfrh.run_ve, axis=1)

In [10]:
dftmp.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/infer_all_19_data_with_best_FEV1.xlsx",
    index=False,
)

In [ ]:
# df.to_excel(
#     dh.get_path_to_main() + "ExcelFiles/CFR/AR_19_data_with_best_FEV1.xlsx",
#     index=False,
# )

# Load variables for associations

In [28]:
names_map = {
    "s01caseid_original": "ID",
    "s02hospivqty": "Hosp IVs",
    "s02hospivoveralltotaldays": "Hosp IV days",
    "s02homeivqty": "Home IVs",
    "s02homeivoveralltotaldays": "Home IV days",
    # I believe these are home orals because these
    "s01coursesoforalantibiotics": "Oral",
    # Technically Non-IV hosp treatments = Orals?
    "s02hospnonivqty": "Non Hosp IVs",
    "s02hospnonivoveralltotaldays": "Non Hosp IV days",
    # complications
    "s06acutechestepisodesqty": "Chest episodes",
    "s06cmpscoughfractqty": "Cough episodes",
    "s06cmpspulmabscessqty": "Pulm Abscess",
    "s09patientsmokes": "Smoking status",
    "s09patientsmokessecondhand": "2nd hand smoking exposure",
}
cols2read = list(names_map.keys())
colnames = list(names_map.values())

df = pd.DataFrame(columns=colnames + ["Date Recorded"])
# years = [2019, 2020, 2021, 2022, 2023]
years = [2019]  # , 2020, 2021, 2022, 2023]
for year in years:
    dftmp = cfrd.load_cfr_data(f"{year}", cols2read, colnames)
    dftmp["Date Recorded"] = datetime.date(year, 1, 1)

    df = pd.concat([df, dftmp], ignore_index=True)

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_21820/1899490785.py:29: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



## Antibiotics

### Data corrections

In [32]:
# All NaN (based on hosp IVs)
# df[df['Hosp IVs'].isna()].ID.to_list()
nan_ids = ["B155992", "B156010", "B159799", "B166659", "B170326", "B171844", "C221154"]
df = df[~dftmp.ID.isin(nan_ids)]

In [33]:
# Correct erroneous values
# df[df['Hosp IV days'] < 0]
df.loc[df.ID == "C223005", "Hosp IV days"] = 0  # instead of -36325.0

df.loc[df.ID == "B161555", "Non Hosp IVs"] = 0
df.loc[df.ID == "B161555", "Non Hosp IV days"] = (
    0  # Instead of 297 (Hosp IV days is already 298)
)

### Data processing

In [42]:
df.ID.nunique()

10063

In [34]:
# Merge home and hosp IV and IV days
df.loc[:, "IVs"] = df["Home IVs"] + df["Hosp IVs"]
df.loc[:, "IV days"] = df["Home IV days"] + df["Hosp IV days"]
df.drop(
    columns=["Home IVs", "Hosp IVs", "Home IV days", "Hosp IV days"],
    inplace=True,
    errors="ignore",
)

In [43]:
# Ensure uniqueness on ID and Date Recorded
if df.duplicated(subset=["ID", "Date Recorded"]).sum() != 0:
    raise ValueError

print("Number of NaN in 'IVs':", df["IVs"].isna().sum())
print("Number of NaN in 'Non Hosp IVs':", df["Non Hosp IVs"].isna().sum())
print("Number of NaN in 'Oral':", df["Oral"].isna().sum())

df.describe().loc[["count", "mean", "std", "min", "max"]]

Number of NaN in 'IVs': 0
Number of NaN in 'Non Hosp IVs': 0
Number of NaN in 'Oral': 130


,Oral,Non Hosp IVs,Non Hosp IV days,Chest episodes,Cough episodes,Pulm Abscess,IVs,IV days
count,9933.000000,10063.000000,10063.000000,50.00000,11.000000,5.000000,10063.000000,10062.000000
mean,2.175375,0.214350,0.627447,1.12000,1.090909,1.400000,1.294445,15.647983
std,2.312708,0.687701,4.130382,0.38545,0.301511,0.894427,2.109329,29.149211
min,0.000000,0.000000,0.000000,1.00000,1.000000,1.000000,0.000000,0.000000
max,71.000000,13.000000,172.000000,3.00000,2.000000,3.000000,20.000000,378.000000


### Abnormally high number of IVs

In [44]:
df[(df["IV days"] + df["Non Hosp IV days"]) > 350][['ID', 'IV days', 'Non Hosp IV days']]

,ID,IV days,Non Hosp IV days
941,B157826,368.0,0.0
3732,B162542,361.0,0.0
4189,B163993,363.0,0.0
7358,B170420,378.0,0.0


### Antibiotics episodes and number of days should be non null together

In [46]:
# Check cases where Oral XOR Oral days are zero
xor_mask = (df["Non Hosp IVs"] == 0) ^ (df["Non Hosp IV days"] == 0)
df[xor_mask][['ID', 'Non Hosp IVs', 'Non Hosp IV days']]

,ID,Non Hosp IVs,Non Hosp IV days
8072,B171434,1.0,0.0


In [47]:
# Check cases where Oral XOR Oral days are zero
xor_mask = (df["IVs"] == 0) ^ (df["IV days"] == 0)
df[xor_mask][['ID', 'IVs', 'IV days']]

,ID,IVs,IV days
10059,C223005,12.0,0.0


In [24]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/IV_data_2019.xlsx",
    index=False,
)

### Plots

#### Plot number of days treatment

Note: rather plot subgroups of individuals where the model confidently disagrees

In [109]:
# Check number of IVs, now showing normalized percentage per bar and showing the percent label per bar
import plotly.express as px


for col in ["IV days", "Non Hosp IV days"]:
    fig = px.histogram(df, x=col, histnorm="percent", text_auto=".2f")
    fig.update_traces(textangle=270)
    ids = df[~df[col].isna()].ID.nunique()
    notnan = (~df[col].isna()).sum()
    title = f"{col} from 2019 registry, {notnan} entries, {ids} IDs"
    fig.update_layout(
        height=300,
        width=800,
        title=dict(text=title, font=dict(size=14)),
        margin=dict(l=40, r=20, t=40, b=40),
        xaxis_title=col,
        yaxis_title="Percent",
    )
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")

# CCL: 75% of individuals had no hosp IVs between 2019 and 2023. 85% no home IVs. Trikata arrived in nov 2020

#### Plot number of IV/Oral episodes

In [111]:
# Check number of IVs, now showing normalized percentage per bar and showing the percent label per bar
import plotly.express as px


# for col in ["Hosp IVs", "Home IVs"]:
for col in ["IVs", "Oral", "Non Hosp IVs"]:
    fig = px.histogram(df, x=col, histnorm="percent", text_auto=".2f")
    fig.update_traces(textangle=270)
    ids = df[~df[col].isna()].ID.nunique()
    notnan = (~df[col].isna()).sum()
    title = f"{col} from 2019 registry, {notnan} entries, {ids} IDs"
    fig.update_layout(
        height=300,
        width=800,
        title=dict(text=title, font=dict(size=14)),
        margin=dict(l=40, r=20, t=40, b=40),
        xaxis_title=col,
        yaxis_title="Percent",
    )
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")

# CCL: 75% of individuals had no hosp IVs between 2019 and 2023. 85% no home IVs. Trikata arrived in nov 2020

#### Older plots

In [ ]:
# Compute avg IVs per year
# skipna is True by default meaning that nans are excluded
df_avg_ivs_per_year = (
    df.groupby("ID")
    .agg(
        {
            "Home IVs": "mean",
            "Hosp IVs": "mean",
            "Oral": "mean",
            "Any antibiotics": "mean",
        }
    )
    .rename(columns={"Home IVs": "Avg Home IVs", "Hosp IVs": "Avg Hosp IVs"})
)

df_avg_ivs_per_year.describe(percentiles=[])
# CCL: 2x more hosp IVs than home IVs
# Mean hosp IVs: 0.5 per year

,Avg Home IVs,Avg Hosp IVs,Oral,Any antibiotics
count,11663.000000,11663.000000,11663.000000,11663.000000
mean,0.284902,0.500503,1.750503,2.535817
std,0.688697,0.922580,1.576887,2.244520
min,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.200000,1.400000,2.000000
max,11.000000,20.000000,14.200000,25.000000


In [117]:
df_avg_ivs_per_year

,ID,Avg Home IVs,Avg Hosp IVs,Oral,Any antibiotics
ID,,,,,
0,B155916,0.2,0.4,0.8,1.4
1,B155917,0.0,0.0,1.4,1.4
2,B155918,0.6,0.2,1.4,2.2
3,B155920,0.0,0.2,0.6,0.8
4,B155921,0.2,0.2,1.8,2.2
...,...,...,...,...,...
11658,C226145,0.0,0.0,0.0,0.0
11659,C226146,0.0,0.0,1.0,1.0
11660,C226147,0.0,2.0,7.0,9.0


In [ ]:
# Melt the dataframe to long format with columns "ID", "Avg", "Type"
df_avg_ivs_long = df_avg_ivs_per_year.melt(
    id_vars="ID",
    value_vars=["Avg Home IVs", "Avg Hosp IVs", "Oral"],
    var_name="Type",
    value_name="Avg",
)
df_avg_ivs_long

,ID,Type,Avg
0,B155916,Avg Home IVs,0.2
1,B155917,Avg Home IVs,0.0
2,B155918,Avg Home IVs,0.6
3,B155920,Avg Home IVs,0.0
4,B155921,Avg Home IVs,0.2
...,...,...,...
34984,C226145,Oral,0.0
34985,C226146,Oral,1.0
34986,C226147,Oral,7.0
34987,C226148,Oral,2.0


In [ ]:
import numpy as np

# Determine the max value for binning
max_val = df_avg_ivs_long["Avg"].max()
# Create bin edges: first bin for 0, then (0,1], (1,2], ..., (N-1,N]
# Ensure max_val+1 to include the rightmost data point
bin_edges = np.concatenate([np.arange(-1, np.ceil(max_val) + 1)])
bin_labels = ["0"] + [f"({i};{i+1}]" for i in range(0, int(np.ceil(max_val)))]

antibio_binned = pd.cut(
    df_avg_ivs_long["Avg"],
    bins=bin_edges,
    labels=bin_labels,
)

fig = px.histogram(
    df_avg_ivs_long.assign(antibio_binned=antibio_binned),
    x="antibio_binned",
    color="Type",
    histnorm="percent",
    text_auto=".1f",
    category_orders={"antibio_binned": bin_labels},
)

# Make the first bar (bin for '0') white
bar_colors = ["grey"] + ["#0072b2"] * (len(bin_labels) - 1)
fig.update_traces(
    textangle=90,
    # marker_color=bar_colors,
    textfont=dict(size=9),
    textposition="outside",
)
fig.update_yaxes(range=[0, 150])
fig.update_xaxes(tickangle=45)

title = f"Demography per antibiotic in 2019-23 registry, {ids} IDs"

fig.update_layout(
    height=300,
    width=800,
    title=dict(text=title, font=dict(size=12)),
    margin=dict(l=40, r=20, t=40, b=40),
    xaxis_title="Average number of antibiotics per year (hosp/home IV or oral)",
    yaxis_title="Proportion",
)
fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [ ]:
import numpy as np

# Determine the max value for binning
max_val = df_avg_ivs_per_year["Any antibiotics"].max()
# Create bin edges: first bin for 0, then (0,1], (1,2], ..., (N-1,N]
# Ensure max_val+1 to include the rightmost data point
bin_edges = np.concatenate([np.arange(-1, np.ceil(max_val) + 1)])
bin_labels = ["0"] + [f"({i};{i+1}]" for i in range(0, int(np.ceil(max_val)))]

antibio_binned = pd.cut(
    df_avg_ivs_per_year["Any antibiotics"],
    bins=bin_edges,
    labels=bin_labels,
)

fig = px.histogram(
    df_avg_ivs_per_year.assign(antibio_binned=antibio_binned),
    x="antibio_binned",
    histnorm="percent",
    text_auto=".1f",
    category_orders={"antibio_binned": bin_labels},
)

# Make the first bar (bin for '0') white
bar_colors = ["grey"] + ["#0072b2"] * (len(bin_labels) - 1)
fig.update_traces(
    textangle=90,
    marker_color=bar_colors,
    textfont=dict(size=9),
    textposition="outside",
)
fig.update_yaxes(range=[0, 29])
fig.update_xaxes(tickangle=45)

title = f"Demography of all antibiotics in 2019-23 registry, {ids} IDs"

fig.update_layout(
    height=300,
    width=800,
    title=dict(text=title, font=dict(size=12)),
    margin=dict(l=40, r=20, t=40, b=40),
    xaxis_title="Average number of antibiotics per year (hosp/home IV or oral)",
    yaxis_title="Proportion",
)
# fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [114]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/anbitiotics_data_19-23.xlsx",
    index=False,
)

In [ ]:
year = 2023
df["Date Recorded"] = datetime.date(year, 1, 1)

In [20]:
rename_dict = dict(zip(cols2read, colnames))
df = df.rename(columns=rename_dict)